<a href="https://colab.research.google.com/github/icervecon/Dividend-Design-UCITS-Funds/blob/main/Dividend_Policies_as_Product_Design_in_UCITS_Funds_REV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file_path = "Research_Funds_20260122_FINAL.xlsx"
df = pd.read_excel(file_path)

# Ver nombres de columnas
print(df.columns.tolist())

Tablas A.1

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Cargar
df = pd.read_excel("Research_Funds_20260122_FINAL.xlsx")

# Limpieza
df['div_value_usd'] = pd.to_numeric(df['div_value_usd'], errors='coerce')

# Variable principal
df['dividend_paid'] = (df['income_dist'] == 'Paid').astype(int)

# Intensidad
df['log_div_value'] = np.log1p(df['div_value_usd'])

# Controles
controls = "log_tna + fund_age_years + is_equity + is_fixed_income"

vars_base = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]

df_base = df[vars_base].dropna()

# -----------------------------
# (1) MODELO BASE
# -----------------------------
model1 = smf.ols(
    f"dividend_paid ~ retail_eligible_i + {controls}",
    data=df_base
).fit()

res1 = model1.get_robustcov_results(
    cov_type='cluster',
    groups=df_base.loc[model1.model.data.row_labels, 'Management_Group']
)

# -----------------------------
# (2) INTENSIDAD (SOLO PAGADORES)
# -----------------------------
df_paid = df_base.copy()
df_paid['log_div_value'] = df['log_div_value']

df_paid = df_paid[
    (df_paid['dividend_paid'] == 1) &
    (df_paid['log_div_value'].notnull())
]

model2 = smf.ols(
    f"log_div_value ~ retail_eligible_i + {controls}",
    data=df_paid
).fit()

res2 = model2.get_robustcov_results(
    cov_type='cluster',
    groups=df_paid.loc[model2.model.data.row_labels, 'Management_Group']
)

# Output
print("\n(1) Dividend (Baseline)")
print(res1.summary())

print("\n(2) Log Dividend Value (Paid Funds)")
print(res2.summary())

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. CARGAR BASE ORIGINAL
# -----------------------------

df = pd.read_excel("Research_Funds_20260122_FINAL.xlsx")

# -----------------------------
# 2. CONSTRUCCIÓN DE VARIABLES
# -----------------------------

# Dummy de dividendos
df['dividend_paid'] = (df['income_dist'] == 'Paid').astype(int)

# Asegurar numérico
df['div_value_usd'] = pd.to_numeric(df['div_value_usd'], errors='coerce')

# Intensidad (log)
df['log_div_value'] = np.log1p(df['div_value_usd'])

# -----------------------------
# 3. (OPCIONAL) FRECUENCIA LIMPIA
# -----------------------------
# Solo si quieres incluirla (no obligatorio)

def count_payments(x):
    if pd.isnull(x):
        return np.nan
    x = str(x)
    if "Every Month" in x:
        return 12
    return len(x.split(','))

df['div_freq'] = df['payment_months'].apply(count_payments)

# -----------------------------
# 4. SELECCIONAR VARIABLES ENTREGABLES
# -----------------------------

cols_keep = [
    'fund_id',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group',
    'dividend_paid',
    'log_div_value',
    'div_freq'
]

df_export = df[cols_keep]

# -----------------------------
# 5. EXPORTAR
# -----------------------------

df_export.to_excel("UCITS_dividend_dataset_transformed.xlsx", index=False)

print("Base exportada correctamente.")

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. CARGAR BASE COMPLETA
# -----------------------------
df_full = pd.read_excel("Research_Funds_20260122_FINAL.xlsx")

# -----------------------------
# 2. DEFINIR MUESTRA FINAL (la de tus regresiones)
# -----------------------------
vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income'
]

# reconstruimos dividend_paid por si no está
df_full['dividend_paid'] = (df_full['income_dist'] == 'Paid').astype(int)

df_final = df_full[vars_model].dropna()

# -----------------------------
# 3. COMPARACIÓN DE MUESTRAS
# -----------------------------
vars_compare = ['log_tna', 'fund_age_years', 'is_equity', 'is_fixed_income']

summary_full = df_full[vars_compare].mean()
summary_final = df_final[vars_compare].mean()

comparison = pd.DataFrame({
    'Full sample': summary_full,
    'Final sample': summary_final,
    'Difference': summary_final - summary_full
})

print(comparison)

In [ ]:
from scipy import stats

for var in ['log_tna', 'fund_age_years', 'is_equity', 'is_fixed_income']:
    tstat, pval = stats.ttest_ind(df_full[var].dropna(), df_final[var].dropna(), equal_var=False)
    print(var, pval)

In [ ]:
# Crear variable primero
df['high_min_inv'] = (
    df['Minimum Investment Initial'] > df['Minimum Investment Initial'].median()
).astype(int)

# Incluir Management_Group en la submuestra
vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'high_min_inv',
    'Management_Group'
]

df_base = df[vars_model].dropna()

In [ ]:
model_alt = smf.ols(
    "dividend_paid ~ retail_eligible_i + high_min_inv + log_tna + fund_age_years + is_equity + is_fixed_income",
    data=df_base
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_base['Management_Group']}
)

print(model_alt.summary())

Tabla A.3

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 1. Cargar datos
df = pd.read_excel("Research_Funds_20260122_FINAL.xlsx")

# 2. Variables necesarias
df['dividend_paid'] = (df['income_dist'] == 'Paid').astype(int)

df['log_tna'] = np.log(df['tna_usd_mil'] + 1)
df['fund_age_years'] = (pd.to_datetime("today") - pd.to_datetime(df['launch_date'])).dt.days / 365

df['high_min_inv'] = (
    df['Minimum Investment Initial'] > df['Minimum Investment Initial'].median()
).astype(int)

# 3. Submuestra
vars_model = [
    'dividend_paid',
    'retail_eligible_i',
    'high_min_inv',
    'log_tna',
    'fund_age_years',
    'is_equity',
    'is_fixed_income',
    'Management_Group'
]

df_base = df[vars_model].dropna()

# 4. Modelo
model_alt = smf.ols(
    "dividend_paid ~ retail_eligible_i + high_min_inv + log_tna + fund_age_years + is_equity + is_fixed_income",
    data=df_base
).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_base['Management_Group']}
)

# 5. Output CLARO
print(model_alt.summary())

In [ ]:
cols_keep = [
    'high_min_inv'
]

df_export = df[cols_keep]

# -----------------------------
# 5. EXPORTAR
# -----------------------------

df_export.to_excel("UCITS_dividend_dataset_transformed.xlsx", index=False)

print("Base exportada correctamente.")